# 10.2 Examples of table-handling statements

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

To transfer information between an AMPL model and a relational table, we begin with
a table declaration that establishes the correspondence between them. Certain details
of this declaration depend on the software being used to create and maintain the table. In
the case of the four-column table of diet data defined above, some of the possibilities are
as follows:
- For a Microsoft Access table in a database file diet.mdb:

```ampl
table Foods IN "ODBC" "diet.mdb":
	FOOD <- [FOOD], cost, f_min, f_max;
```

- For a Microsoft Excel range from a workbook file diet.xls:

```ampl
table Foods IN "ODBC" "diet.xls":
	FOOD <- [FOOD], cost, f_min, f_max;
```

- For an ASCII text table in file Foods.tab:

```ampl
table Foods IN:
	FOOD <- [FOOD], cost, f_min, f_max;
```

Each table declaration has two parts. Before the colon, the declaration provides general
information. First comes the table name — Foods in the examples above — which
will be the name by which the table is known within AMPL. The keyword IN states that
the default for all non-key table columns will be read-only; AMPL will read values in
from these columns and will not write out to them.

Details for locating the table in an external database file are provided by the character
strings such as "ODBC" and "diet.mdb", with the AMPL table name (Foods) providing
a default where needed:
- For Microsoft Access, the table is to be read from database file
	diet.mdb using AMPL's ODBC handler. The table's name within the database
	file is taken to be Foods by default.
- For Microsoft Excel, the table is to be read from spreadsheet file
	diet.xls using AMPL's ODBC handler. The spreadsheet range containing
	the table is taken to be Foods by default.
- Where no details are given, the table is read by default from the ASCII text
	file Foods.tab using AMPL's built-in text table handler.
**[Figure 10-2](#fig-10-2):** Access relational table.

In general, the format of the character strings in the table declaration depends upon the
table handler being used. The strings required by the handlers used in our examples are
described briefly in [Section 10.7](./10_7_examples_of_table_handlers.ipynb), and in detail in online documentation for specific table
handlers.

After the colon, the table declaration gives the details of the correspondence
between AMPL entities and relational table columns. The four comma-separated entries
correspond to four columns in the table, starting with the key column distinguished by
surrounding brackets [...]. In this example, the names of the table columns (`FOOD`,
`cost`, `f_min`, `f_max`) are the same as the names of the corresponding AMPL components.
The expression `FOOD <- [FOOD]` indicates that the entries in the key column
FOOD are to be copied into AMPL to define the members of the set FOOD.

The table declaration only defines a correspondence. To read values from columns
of a relational table into AMPL sets and parameters, it is necessary to give an explicit

```ampl
read table
```

command.

Thus, if the data values were in an Access relational table like [Figure 10-2](#fig-10-2), the
table declaration for Access could be used together with the `read table` command
to read the members of FOOD and values of cost, f_min and f_max into the corresponding
AMPL set and parameters:
**[Figure 10-3](#fig-10-3):** Excel worksheet range.

```ampl
ampl:   model diet.mod;
ampl:   table Foods IN "ODBC" "diet.mdb":
ampl?      FOOD <- [FOOD], cost, f_min, f_max;
ampl:   read table Foods;
ampl:   display cost, f_min, f_max;
:        cost f_min f_max    :=
BEEF     3.19    2    10
CHK      2.59    2    10
FISH     2.29    2    10
HAM      2.89    2    10
MCH      1.89    2    10
MTL      1.99    2    10
SPG      1.99    2    10
TUR      2.49    2    10
;
```

(The `display` command confirms that the database values were read as intended.) If
the data values were instead in an Excel worksheet range like [Figure 10-3](#fig-10-3), the values
would be read in the same way, but using the table declaration for Excel:

```ampl
ampl: model diet.mod;
 ampl: table Foods IN "ODBC" "diet.xls":
 ampl?    FOOD <- [FOOD], cost, f_min, f_max;
 ampl: read table Foods;
```

And if the values were in a file Foods.tab containing a text table like this:

```ampl
ampl.tab 1 3
FOOD    cost    f_min  f_max
BEEF    3.19    2      10
CHK     2.59    2      10
FISH    2.29    2      10
HAM     2.89    2      10
MCH     1.89    2      10
MTL     1.99    2      10
SPG     1.99    2      10
TUR     2.49    2      10
```

the declaration for a text table would be used:

```ampl
ampl: model diet.mod;
ampl: table Foods IN: FOOD <- [FOOD], cost, f_min, f_max;
ampl: read table Foods;
```

Because the AMPL table name Foods is the same in all three of these examples, the
`read table` command is the same for all three: `read table` Foods. In general, the
`read table` command only specifies the AMPL name of the table to be read. All information
about what is to be read, and how it is to be handled, is taken from the named
table's definition in the table declaration.

To create the second (7-column) relational table example of the previous section, we
could use a pair of table declarations:

```ampl
table ImportFoods IN "ODBC" "diet.mdb" "Foods":
   FOOD <- [FOOD], cost, f_min, f_max;
table ExportFoods OUT "ODBC" "diet.mdb" "Foods":
   FOOD <- [FOOD], Buy, Buy.rc ˜ BuyRC,
   {j in FOOD} Buy[j]/f_max[j] ˜ BuyFrac;
```

or a single table declaration combining the input and output information:

```ampl
table Foods "ODBC" "diet.mdb": [FOOD] IN, cost IN,
   f_min IN, f_max IN, Buy OUT, Buy.rc ˜ BuyRC OUT,
   {j in FOOD} Buy[j]/f_max[j] ˜ BuyFrac OUT;
```

These examples show how the AMPL table name (such as `ExportFoods`) may be different
from the name of the corresponding table within the external file (as indicated by
the subsequent string "Foods"). A number of other useful options are also seen here:
IN and OUT are associated with individual columns of the table, rather than with the
whole table; [FOOD] IN is used as an abbreviation for FOOD <- [FOOD]; columns of
the table are associated with the values of variables `Buy` and expressions `Buy.rc` and
Buy[j]/f_max[j]; `Buy.rc` ˜ `BuyRC` and `{j in FOOD} Buy[j]/f_max[j]` ˜
`BuyFrac` associate an AMPL expression (to the left of the ˜ operator) with a database
column heading (to the right).

To write meaningful results back to the Access database, we need to read all of the
diet model's data, then solve, and then give a `write table` command. Here's how it
all might look using separate table declarations to read and write the Access table
Foods:

```ampl
ampl:   model diet.mod;
ampl:   table ImportFoods IN "ODBC" "diet.mdb" "Foods":
ampl?      FOOD <- [FOOD], cost, f_min, f_max;
ampl:   table Nutrs IN "ODBC" "diet.mdb": NUTR <- [NUTR],
ampl?      n_min, n_max;
ampl:   table Amts IN "ODBC" "diet.mdb": [NUTR, FOOD], amt;
ampl:   read table ImportFoods;
ampl:   read table Nutrs;
ampl:   read table Amts;
ampl:   solve;
ampl:   table ExportFoods OUT "ODBC" "diet.mdb" "Foods":
ampl?      FOOD <- [FOOD],
ampl?      Buy, Buy.rc ˜ BuyRC,
ampl?      {j in FOOD} Buy[j]/f_max[j] ˜ BuyFrac;
ampl:   write table ExportFoods;
```

and here is an alternative using a single declaration to both read and write Foods:

```ampl
ampl:   model diet.mod;
ampl:   table Foods "ODBC" "diet.mdb":
ampl?      [FOOD] IN, cost IN, f_min IN, f_max IN,
ampl?      Buy OUT, Buy.rc ˜ BuyRC OUT,
ampl?      {j in FOOD} Buy[j]/f_max[j] ˜ BuyFrac OUT;
ampl:   table Nutrs IN "ODBC" "diet.mdb":
ampl?      NUTR <- [NUTR], n_min, n_max;
ampl:   table Amts IN "ODBC" "diet.mdb": [NUTR, FOOD], amt;
ampl:   read table Foods;
ampl:   read table Nutrs;
ampl:   read table Amts;
ampl:   solve;
ampl:   write table Foods;
```

Either way, the Access table Foods would end up having three additional columns, as
seen in [Figure 10-4](#fig-10-4).

The same operations are handled similarly for other types of database files. In general,
the actions of a `write table` command are determined by the previously declared
AMPL table named in the command, and by the status of the external file associated with
the AMPL table through its table declaration. Depending on the circumstances, the
`write table` command may create a new external file or table, overwrite an existing
table, overwrite certain columns within an existing table, or append columns to an existing
table.

The table declaration is the same for multidimensional AMPL entities, except that
there must be more than one key column specified between brackets [ and ]. For the
steel production example discussed previously, the correspondence to a relational table
could be set up like this:
**[Figure 10-4](#fig-10-4):** Access relational table with output columns.

```ampl
table SteelProd "ODBC" "steel.mdb":
   [PROD, TIME], market IN, revenue IN,
   Make OUT, Sell OUT, Inv OUT;
```

Here the key columns PROD and TIME are not specified as IN. This is because the
parameters to be read in, market and revenue, are indexed in the AMPL model over
the set {PROD, 1..T}, whose membership would be specified by use of other, simpler
tables. The `read table` SteelProd command merely uses the PROD and TIME
entries of each database row to determine the pair of indices (subscripts) that are to be
associated with the market and revenue entries in the row.

Our transportation example also involves a relational table for two-dimensional entities
, and the associated table declaration is similar:

```ampl
table TransLinks "ODBC" "trans.xls" "Links":
   LINKS <- [ORIG, DEST], cost IN, Trans OUT;
```

The difference here is that LINKS, the AMPL set of pairs over which cost and Trans
are indexed, is part of the data rather than being determined from simpler sets or parameters.
Thus we write LINKS <- [ORIG, DEST], to request that pairs from the key
columns be read into LINKS at the same time that the corresponding values are read into
cost. This distinction is discussed further in the next section.

As you can see from even our simple examples so far, table statements tend to be
cumbersome to type interactively. Instead they are usually placed in AMPL programs, or
scripts, which are executed as described in [Chapter 13](#missing) <!--- xTODO missing reference --->. The `read table` and write
table statements may be included in the scripts as well. You can define a table and
then immediately read or write it, as seen in some of our examples, but a script is often
more readable if the complex table statements are segregated from the statements that
read and write the tables.
The rest of this chapter will concentrate on table statements. Complete sample
scripts and Access or Excel files for the diet, production, and transportation examples can
be obtained from the AMPL web site.